<a href="https://colab.research.google.com/github/deepanrajm/deep_learning/blob/master/XAI/Explainable_AI_CAM_SHAP_LIME.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Explainable AI (XAI): CAM, SHAP and LIME

## 0. Setup

In [ ]:
!pip -q install grad-cam shap lime xgboost ultralytics
import os, urllib.request, itertools, math, copy, warnings
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import torch, torch.nn.functional as F
warnings.filterwarnings('ignore')
device = 'cuda' if torch.cuda.is_available() else 'cpu'
np.random.seed(42); torch.manual_seed(42)
print('Device:', device)

In [ ]:
# --- Download test images ---
os.makedirs('images', exist_ok=True)
IMAGES = {
    'cat_dog': 'https://raw.githubusercontent.com/jacobgil/pytorch-grad-cam/master/examples/both.png',
    'dog':     'https://raw.githubusercontent.com/EliSchwartz/imagenet-sample-images/master/n02099601_golden_retriever.JPEG',
    'bus':     'https://raw.githubusercontent.com/ultralytics/ultralytics/main/ultralytics/assets/bus.jpg',
}
for k, u in IMAGES.items():
    p = f'images/{k}.jpg'
    if not os.path.exists(p): urllib.request.urlretrieve(u, p)
from PIL import Image
fig, ax = plt.subplots(1, 3, figsize=(14, 4))
for a, k in zip(ax, IMAGES):
    a.imshow(Image.open(f'images/{k}.jpg').convert('RGB')); a.set_title(k); a.axis('off')
plt.show()

## Part 1 — Intrinsic (glass-box) explanations
Some models explain themselves. Dataset: **Breast Cancer Wisconsin** (built into scikit-learn, 30 features, malignant vs benign).

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance

bc = load_breast_cancer(as_frame=True)
Xb, yb = bc.data, bc.target          # target: 1 = benign, 0 = malignant
Xb_tr, Xb_te, yb_tr, yb_te = train_test_split(Xb, yb, test_size=0.25, random_state=42, stratify=yb)
scaler = StandardScaler().fit(Xb_tr)

# (a) Logistic regression: coefficients on standardized features = global importance + direction
logreg = LogisticRegression(max_iter=2000).fit(scaler.transform(Xb_tr), yb_tr)
coef = pd.Series(logreg.coef_[0], index=Xb.columns).sort_values()
print('Logistic regression accuracy:', round(logreg.score(scaler.transform(Xb_te), yb_te), 3))
pd.concat([coef.head(6), coef.tail(6)]).plot.barh(figsize=(8, 5), color=['tab:red'] * 6 + ['tab:green'] * 6)
plt.title('Logistic regression coefficients (red → malignant, green → benign)'); plt.show()

In [ ]:
# (b) A shallow decision tree is readable as if-then rules
tree = DecisionTreeClassifier(max_depth=3, random_state=0).fit(Xb_tr, yb_tr)
print('Decision tree accuracy:', round(tree.score(Xb_te, yb_te), 3))
plt.figure(figsize=(18, 7)); plot_tree(tree, feature_names=Xb.columns, class_names=['malignant', 'benign'], filled=True, fontsize=9); plt.show()

In [ ]:
# (c) Permutation importance: shuffle one feature and see how much accuracy drops (model-agnostic, global)
rf = RandomForestClassifier(n_estimators=300, random_state=0).fit(Xb_tr, yb_tr)
pi = permutation_importance(rf, Xb_te, yb_te, n_repeats=15, random_state=0)
order = pi.importances_mean.argsort()[-10:]
plt.figure(figsize=(8, 5)); plt.boxplot(pi.importances[order].T, vert=False, labels=Xb.columns[order])
plt.title(f'Permutation importance (Random Forest, acc={rf.score(Xb_te, yb_te):.3f})'); plt.xlabel('accuracy drop when shuffled'); plt.show()

## Part 2 — CAM family: where is the CNN looking?
Model: **ResNet-50** pretrained on ImageNet. The last conv block (`layer4`) outputs 2048 feature maps of 7×7.

**Grad-CAM recipe:** forward → pick class score `y_c` → backprop to `layer4` → average gradients per map = weight `α_k` → weighted sum of maps → ReLU → upsample → overlay.

In [ ]:
from torchvision.models import resnet50, ResNet50_Weights
weights = ResNet50_Weights.IMAGENET1K_V2
model = resnet50(weights=weights).eval().to(device)
LABELS = weights.meta['categories']
MEAN, STD = np.array([0.485, 0.456, 0.406]), np.array([0.229, 0.224, 0.225])

def load_img(path, size=224):
    img = np.array(Image.open(path).convert('RGB').resize((size, size))) / 255.0   # float RGB 0..1
    x = torch.tensor(((img - MEAN) / STD).transpose(2, 0, 1), dtype=torch.float32)[None].to(device)
    return img, x

def top5(x):
    with torch.no_grad(): p = model(x).softmax(-1)[0]
    v, i = p.topk(5); return [(int(j), LABELS[j], float(q)) for q, j in zip(v.cpu(), i.cpu())]

img_cd, x_cd = load_img('images/cat_dog.jpg')
for idx, name, p in top5(x_cd): print(f'{idx:4d}  {name:<22s} {p:.1%}')

### 2.1 Grad-CAM

In [ ]:
def gradcam_scratch(model, x, target_class, layer):
    store = {}
    def fwd_hook(module, inp, out):
        store['act'] = out
        out.register_hook(lambda g: store.__setitem__('grad', g))     # capture gradient on backward
    h = layer.register_forward_hook(fwd_hook)
    model.zero_grad()
    score = model(x)[0, target_class]          # class score before softmax
    score.backward()
    h.remove()
    A, G = store['act'][0], store['grad'][0]   # (C, h, w)
    alpha = G.mean(dim=(1, 2))                 # global-average-pool the gradients -> one weight per map
    cam = F.relu((alpha[:, None, None] * A).sum(0))
    cam = F.interpolate(cam[None, None], size=x.shape[-2:], mode='bilinear', align_corners=False)[0, 0]
    cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
    return cam.detach().cpu().numpy()

def overlay(img, cam, alpha=0.45):
    heat = plt.cm.jet(cam)[..., :3]
    return np.clip((1 - alpha) * img + alpha * heat, 0, 1)

CAT, DOG = 281, 243      # ImageNet: 281 'tabby cat', 243 'bull mastiff'
fig, ax = plt.subplots(1, 3, figsize=(15, 5))
ax[0].imshow(img_cd); ax[0].set_title('Input')
for a, c in zip(ax[1:], [CAT, DOG]):
    a.imshow(overlay(img_cd, gradcam_scratch(model, x_cd, c, model.layer4))); a.set_title(f'Grad-CAM for "{LABELS[c]}"')
for a in ax: a.axis('off')
plt.suptitle('Same image, two classes → two different explanations', fontsize=13); plt.show()

### 2.2 The CAM family
* **Grad-CAM++** – better with multiple instances · **Score-CAM** – gradient-free (masks the input with each map)
* **XGrad-CAM / LayerCAM** – refined weighting · **EigenCAM** – first principal component, class-agnostic, no gradients

In [ ]:
from pytorch_grad_cam import GradCAM, GradCAMPlusPlus, ScoreCAM, XGradCAM, LayerCAM, EigenCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image

methods = {'Grad-CAM': GradCAM, 'Grad-CAM++': GradCAMPlusPlus, 'XGrad-CAM': XGradCAM,
           'LayerCAM': LayerCAM, 'Score-CAM': ScoreCAM, 'EigenCAM': EigenCAM}
target_layers = [model.layer4[-1]]
fig, ax = plt.subplots(1, len(methods), figsize=(4 * len(methods), 4.5))
for a, (name, M) in zip(ax, methods.items()):
    with M(model=model, target_layers=target_layers) as cam:
        if name == 'Score-CAM': cam.batch_size = 64
        g = cam(input_tensor=x_cd, targets=[ClassifierOutputTarget(DOG)])[0]
    a.imshow(show_cam_on_image(img_cd.astype(np.float32), g, use_rgb=True)); a.set_title(name); a.axis('off')
plt.suptitle(f'CAM methods for class "{LABELS[DOG]}"'); plt.show()

In [ ]:
# Which layer? Earlier layers = sharper but less class-specific; layer4 = semantic but coarse (7x7)
layers = {'layer2': model.layer2[-1], 'layer3': model.layer3[-1], 'layer4': model.layer4[-1]}
img_dog, x_dog = load_img('images/dog.jpg')
pred_dog = top5(x_dog)[0]; print('Prediction:', pred_dog)
fig, ax = plt.subplots(1, 4, figsize=(16, 4))
ax[0].imshow(img_dog); ax[0].set_title('Input')
for a, (n, L) in zip(ax[1:], layers.items()):
    with GradCAM(model=model, target_layers=[L]) as cam:
        g = cam(input_tensor=x_dog, targets=[ClassifierOutputTarget(pred_dog[0])])[0]
    a.imshow(show_cam_on_image(img_dog.astype(np.float32), g, use_rgb=True)); a.set_title(f'Grad-CAM @ {n}')
for a in ax: a.axis('off')
plt.show()

## Part 3 — SHAP (SHapley Additive exPlanations)
Features are **players**, the prediction is the **payout**; Shapley values split the payout fairly.

`prediction = base_value + Σ φᵢ`  — always adds up exactly.

### 3.1 Shapley values from scratch

In [ ]:
import shap
from sklearn.datasets import load_diabetes
from sklearn.ensemble import GradientBoostingRegressor

dia = load_diabetes(as_frame=True)
feats4 = ['bmi', 'bp', 's5', 'age']                    # 4 features -> 2^4 = 16 coalitions, easy to enumerate
Xd, yd = dia.data[feats4], dia.target
gbr = GradientBoostingRegressor(random_state=0).fit(Xd, yd)
background = Xd.sample(50, random_state=0).values
x0 = Xd.iloc[0].values

def value(S):
    """v(S): average prediction when features in S come from x0 and the rest from background rows."""
    Z = background.copy(); Z[:, list(S)] = x0[list(S)]
    return gbr.predict(Z).mean()

M = len(feats4); phi = np.zeros(M)
for i in range(M):
    others = [j for j in range(M) if j != i]
    for k in range(M):
        for S in itertools.combinations(others, k):
            weight = math.factorial(len(S)) * math.factorial(M - len(S) - 1) / math.factorial(M)
            phi[i] += weight * (value(S + (i,)) - value(S))

lib = shap.explainers.Exact(gbr.predict, shap.maskers.Independent(background, max_samples=50))(x0[None])
print(pd.DataFrame({'from scratch': phi, 'shap library': lib.values[0]}, index=feats4).round(3))
print(f'\nbase value {value(()):.2f} + sum(phi) {phi.sum():.2f} = {value(()) + phi.sum():.2f}   |   model prediction {gbr.predict(x0[None])[0]:.2f}')

### 3.2 TreeSHAP on a real problem: who earns more than $50K? (UCI Adult income)

In [ ]:
X, y = shap.datasets.adult()                    # numeric-encoded features
X_disp, _ = shap.datasets.adult(display=True)   # human-readable categories for plots
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=7)

import xgboost as xgb
clf = xgb.XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.1, subsample=0.8, eval_metric='logloss').fit(X_tr, y_tr)
print('Test accuracy:', (clf.predict(X_te) == y_te).mean().round(3))
X.head()

In [ ]:
explainer = shap.TreeExplainer(clf)
X_sample = X_te.iloc[:1000]
sv = explainer(X_sample)                        # Explanation object, values in log-odds
sv.display_data = X_disp.loc[X_sample.index].values
# additivity check
margin = clf.predict(X_sample.iloc[:1], output_margin=True)[0]
print(f'base {sv.base_values[0]:.3f} + sum(SHAP) {sv.values[0].sum():.3f} = {sv.base_values[0] + sv.values[0].sum():.3f}  vs model log-odds {margin:.3f}')

#### Local explanation: why this person? (waterfall + force plot)

In [ ]:
person = int(np.argmin(np.abs(clf.predict_proba(X_sample)[:, 1] - 0.75)))   # someone the model rates ~75% likely >50K
print('P(>50K) =', clf.predict_proba(X_sample.iloc[[person]])[0, 1].round(3))
shap.plots.waterfall(sv[person], max_display=10)

#### Global explanation: beeswarm, bar, dependence

In [ ]:
shap.plots.beeswarm(sv, max_display=12)       # each dot = one person; colour = feature value; x = impact

In [ ]:
shap.plots.bar(sv, max_display=12)            # mean |SHAP| = global importance

In [ ]:
shap.plots.scatter(sv[:, 'Age'], color=sv[:, 'Education-Num'])   # non-linear effect of age + interaction

### 3.3 SHAP for text (a transformer sentiment model)

In [ ]:
from transformers import pipeline
sentiment = pipeline('sentiment-analysis', model='distilbert-base-uncased-finetuned-sst-2-english', top_k=None,
                     device=0 if device == 'cuda' else -1)
reviews = ['The battery life is amazing but the camera is honestly disappointing.',
           'Worst customer service ever, although delivery was quick.']
text_explainer = shap.Explainer(sentiment)
sv_text = text_explainer(reviews)
shap.plots.text(sv_text[:, :, 'POSITIVE'])

## Part 4 — LIME (Local Interpretable Model-agnostic Explanations)
1. Perturb the input · 2. Ask the black box · 3. Weight samples by closeness · 4. Fit a small linear model · 5. Read its coefficients.

### 4.1 LIME on tabular data (same income model and same person)

In [ ]:
from lime.lime_tabular import LimeTabularExplainer
cat_cols = ['Workclass', 'Marital Status', 'Occupation', 'Relationship', 'Race', 'Sex', 'Country']
lime_tab = LimeTabularExplainer(X_tr.values, feature_names=list(X.columns), class_names=['<=50K', '>50K'],
                                categorical_features=[X.columns.get_loc(c) for c in cat_cols if c in X.columns],
                                discretize_continuous=True, mode='classification', random_state=42)
exp = lime_tab.explain_instance(X_sample.iloc[person].values, clf.predict_proba, num_features=8, num_samples=5000)
print('Local surrogate R² (fidelity):', round(exp.score, 3))
exp.as_pyplot_figure(); plt.title('LIME: why >50K for this person'); plt.tight_layout(); plt.show()

In [ ]:
exp.show_in_notebook(show_table=True)

#### LIME stability check — sampling is random, so explanations can change between runs

In [ ]:
runs = []
for seed in range(5):
    e = LimeTabularExplainer(X_tr.values, feature_names=list(X.columns), class_names=['<=50K', '>50K'],
                             categorical_features=[X.columns.get_loc(c) for c in cat_cols if c in X.columns],
                             discretize_continuous=True, random_state=seed)
    runs.append(dict(e.explain_instance(X_sample.iloc[person].values, clf.predict_proba, num_features=6, num_samples=500).as_list()))
stab = pd.DataFrame(runs).T
stab.columns = [f'seed {s}' for s in range(5)]
print('Weights per run (NaN = feature not in top-6 that run):'); stab.round(3)

### 4.2 LIME on text

In [ ]:
from lime.lime_text import LimeTextExplainer
def predict_text(texts):
    out = sentiment(list(texts), batch_size=64)
    return np.array([[d['score'] for d in sorted(o, key=lambda d: d['label'])] for o in out])   # [NEGATIVE, POSITIVE]

lime_text = LimeTextExplainer(class_names=['NEGATIVE', 'POSITIVE'], random_state=42)
exp_t = lime_text.explain_instance(reviews[0], predict_text, num_features=8, num_samples=500)
exp_t.show_in_notebook(text=True)

### 4.3 LIME on images (superpixels)
The image is split into superpixels; LIME hides random subsets and learns which superpixels support the class.

In [ ]:
from lime import lime_image
from skimage.segmentation import mark_boundaries

def predict_images(batch):                       # batch: (N, H, W, 3) floats 0..1
    x = torch.tensor(((batch - MEAN) / STD).transpose(0, 3, 1, 2), dtype=torch.float32).to(device)
    with torch.no_grad(): return model(x).softmax(-1).cpu().numpy()

lime_img = lime_image.LimeImageExplainer(random_state=42)
exp_i = lime_img.explain_instance(img_cd, predict_images, labels=(CAT, DOG), top_labels=None,
                                  hide_color=0, num_samples=1000, batch_size=64)
fig, ax = plt.subplots(1, 3, figsize=(15, 5))
ax[0].imshow(mark_boundaries(img_cd, exp_i.segments)); ax[0].set_title(f'{exp_i.segments.max() + 1} superpixels')
for a, c in zip(ax[1:], [CAT, DOG]):
    t, m = exp_i.get_image_and_mask(c, positive_only=True, num_features=6, hide_rest=False)
    a.imshow(mark_boundaries(t, m, color=(1, 1, 0))); a.set_title(f'LIME for "{LABELS[c]}"')
for a in ax: a.axis('off')
plt.show()

## Part 5 — Compare and sanity-check
### 5.1 Same image: Grad-CAM vs LIME

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 5))
ax[0].imshow(overlay(img_cd, gradcam_scratch(model, x_cd, DOG, model.layer4))); ax[0].set_title('Grad-CAM (gradients, 1 pass)')
t, m = exp_i.get_image_and_mask(DOG, positive_only=True, num_features=6, hide_rest=True)
ax[1].imshow(t); ax[1].set_title('LIME (1000 perturbations)')
for a in ax: a.axis('off')
plt.suptitle(f'Two methods, one question: why "{LABELS[DOG]}"?'); plt.show()